## 16. TRIGON for graph classification (PROTEINS)

The base TRIGON paper targets node classification. Graph classification is a different setting: we receive a *set* of graphs, each with a single label, and we want to predict the label of a held-out graph. Adapting TRIGON requires a few design choices.

### Dataset

PROTEINS (1113 graphs, average ~39 nodes, 2 classes: enzyme vs non-enzyme). Each node represents an amino acid residue and has a low-dimensional one-hot feature vector encoding its type.

We previously tried MUTAG (188 graphs). MUTAG's test split contains only about 19 graphs, which gave a ± 10-14 point standard deviation across seeds which is too noisy to draw conclusions. PROTEINS is about 6x larger and the test set per split is ~111 graphs, which typically reduces the seed-to-seed variance to roughly ±2-3 points and makes the TRIGON vs baseline comparison actually more informative.

### Design choices and how they differ from node TRIGON

1. **Per-graph views.** k-NN, Delaunay and the original graph are computed separately for each graph in the dataset. UMAP is not suitable for graphs this small, so we use PCA for the Delaunay projection; k-NN uses `k = min(k_knn, n - 1)` to handle graphs with very few nodes gracefully.

2. **Triangle supervision without node labels.** TRIGON's `L_contr` relies on node labels to define `y_tri = 1 iff >= 2 of 3 nodes share the same label`. In graph classification there are no node labels. We use the **argmax of the one-hot node features** (residue types for PROTEINS) as pseudo-labels: a triangle is "positive" iff at least two of its three nodes share the same residue type. This lets `L_contr` push the selector toward type-coherent motifs without touching the graph label.

3. **`L_part` is dropped.** It was per-class node participation, which makes no sense without node labels. We keep only `L_contr`, `L_struct`, and the classification loss `L_gnn`.

4. **Readout and classifier.** After the GCN message passing on `G*`, we sum-pool node embeddings to get a graph embedding, then apply a 2-layer MLP classifier. This is the standard graph-classification recipe and is not part of TRIGON's node-classif pipeline.

5. **Batching.** We use PyTorch Geometric's `DataLoader` which concatenates graphs into a single big disjoint graph per batch. Triangle extraction runs per graph (before batching); the cached triangles are then remapped to batched indices at training time via a `CustomData` subclass.

6. **Train / val / test splits.** PROTEINS has no canonical split; we use 80/10/10 stratified by label, varied across seeds.

### Disclaimer

- We are not reproducing a specific published number. The authors of TRIGON did not publish a graph-classification variant, so there is no official target. Typical GCN numbers on PROTEINS in the literature are around 72-76%.
- This is an adaptation, not a reproduction, several design choices (pseudo-labels from feature argmax, dropping `L_part`, adding readout) are ours.
- Even with PROTEINS' larger test set, TRIGON vs baseline differences of less than a few points should be treated as indistinguishable. Which will be the case. And the most important part than we mention earlier but it was not possible for us to reproduce the TRIGON paper's results. Mostly because the precise Delaunay implementation was not detailed.

In [1]:
# Cell 16.1 — install (run once per session)
!pip install torch_geometric -q
!pip install scikit-learn scipy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.5 MB/s eta 0:00:00


In [2]:
# Cell 16.2 — TRIGON for graph classification on MUTAG (full version)
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from scipy.spatial import Delaunay
from scipy.spatial import QhullError

from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_add_pool
from torch_geometric.utils import to_undirected, dense_to_sparse


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- Custom Data class so DataLoader batches the triangle tensors correctly ---
class CustomData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == "cand_triangles":
            return self.num_nodes
        return super().__inc__(key, value, *args, **kwargs)

    def __cat_dim__(self, key, value, *args, **kwargs):
        if key in ("cand_triangles", "cand_tri_labels"):
            return 0
        return super().__cat_dim__(key, value, *args, **kwargs)


# --- View builders ---
def build_knn_small(x, k):
    n = x.size(0)
    if n <= 1:
        return torch.empty((2, 0), dtype=torch.long)
    k_eff = min(k, n - 1)
    adj = kneighbors_graph(x.detach().cpu().numpy(), n_neighbors=k_eff,
                           mode="connectivity", include_self=False)
    edge_index, _ = dense_to_sparse(torch.tensor(adj.toarray(), dtype=torch.float32))
    return to_undirected(edge_index, num_nodes=n)


def build_delaunay_small(x):
    n = x.size(0)
    if n < 3:
        return torch.empty((2, 0), dtype=torch.long)
    rng = np.random.RandomState(0)
    x_np = x.detach().cpu().numpy() + rng.normal(scale=1e-3, size=(n, x.size(1)))
    if x_np.shape[1] >= 2:
        x_np = PCA(n_components=2).fit_transform(x_np)
    try:
        tri = Delaunay(x_np, qhull_options="QJ")
    except QhullError:
        return torch.empty((2, 0), dtype=torch.long)
    edges = set()
    for s in tri.simplices:
        for i, j in [(0, 1), (1, 2), (0, 2)]:
            a, b = int(s[i]), int(s[j])
            edges.add((min(a, b), max(a, b)))
    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
    ei = torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    return to_undirected(ei, num_nodes=n)


def enumerate_triangles(edge_index, num_nodes):
    if edge_index.numel() == 0:
        return []
    nbrs = [set() for _ in range(num_nodes)]
    for u, v in edge_index.t().tolist():
        if u != v:
            nbrs[u].add(v)
    tris = []
    for u in range(num_nodes):
        for v in nbrs[u]:
            if v <= u:
                continue
            for w in (nbrs[u] & nbrs[v]):
                if w > v:
                    tris.append((u, v, w))
    return tris


def triangle_pseudo_label(tris_t, atom_types):
    if tris_t.numel() == 0:
        return torch.empty((0,), dtype=torch.float)
    li = atom_types[tris_t[:, 0]]
    lj = atom_types[tris_t[:, 1]]
    lk = atom_types[tris_t[:, 2]]
    distinct = (li != lj) & (lj != lk) & (li != lk)
    return (~distinct).float()


def precompute_candidates_for_dataset(dataset, k_knn=5, verbose=True):
    total = 0
    for d in dataset:
        n = d.num_nodes
        e_o = to_undirected(d.edge_index, num_nodes=n)
        e_k = build_knn_small(d.x, k_knn)
        e_d = build_delaunay_small(d.x)
        tris = list(set(enumerate_triangles(e_o, n)
                        + enumerate_triangles(e_k, n)
                        + enumerate_triangles(e_d, n)))
        d.cand_triangles = (torch.tensor(tris, dtype=torch.long)
                            if tris else torch.empty((0, 3), dtype=torch.long))
        d.cand_tri_labels = triangle_pseudo_label(d.cand_triangles, d.x.argmax(1))
        total += d.cand_triangles.size(0)
    if verbose:
        print(f"Precomputed {total} candidate triangles across "
              f"{len(dataset)} graphs (avg {total / max(len(dataset), 1):.1f} per graph).")


# --- Modules ---
class TriangleEncoder(nn.Module):
    def __init__(self, in_d, hidden=64, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(3 * in_d, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        h = F.relu(self.fc2(h))
        return h


class TriangleSelector(nn.Module):
    def __init__(self, in_d, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_d, 32), nn.LayerNorm(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 16), nn.LayerNorm(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 2),
        )
        self.tau_raw = nn.Parameter(torch.tensor(0.5413))  # softplus ~= 1.0

    @property
    def tau(self):
        return F.softplus(self.tau_raw) + 1e-4

    def forward(self, g, hard=True):
        logits = self.net(g)
        probs = F.gumbel_softmax(logits, tau=self.tau, hard=hard)
        return probs[:, 1]


class TrigonGraphClassifier(nn.Module):
    def __init__(self, in_d, hidden, out_d, num_layers=3, dropout=0.3):
        super().__init__()
        assert num_layers >= 2
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_d, hidden))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden, hidden))
        self.dropout = dropout
        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_d),
        )

    def forward(self, x, ei, batch):
        h = x
        for conv in self.convs:
            h = conv(h, ei)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        pooled = global_add_pool(h, batch)
        return self.cls(pooled)


# --- Losses ---
def contrastive_triangle_loss_gc(p, y_tri):
    if p.numel() == 0:
        return torch.zeros((), device=p.device)
    pos = (1.0 - y_tri) * p.pow(2)
    neg = y_tri * torch.clamp(1.0 - p, min=0.0)
    return (pos + neg).mean()


def structural_loss_gc(selected, node_feats):
    if selected.numel() == 0:
        return torch.zeros((), device=node_feats.device)
    xi = node_feats[selected[:, 0]]
    xj = node_feats[selected[:, 1]]
    xk = node_feats[selected[:, 2]]
    return (torch.linalg.norm(xi - xj, dim=1)
            + torch.linalg.norm(xj - xk, dim=1)
            + torch.linalg.norm(xk - xi, dim=1)).mean()


# --- Batch helpers ---
def gather_batch_triangles(batch):
    if not hasattr(batch, "cand_triangles") or batch.cand_triangles.numel() == 0:
        return (torch.empty((0, 3), dtype=torch.long, device=batch.x.device),
                torch.empty((0, 3 * batch.x.size(1)), device=batch.x.device),
                torch.empty((0,), device=batch.x.device))
    tris = batch.cand_triangles
    feats = torch.cat([batch.x[tris[:, 0]], batch.x[tris[:, 1]], batch.x[tris[:, 2]]], dim=1)
    return tris, feats, batch.cand_tri_labels


def triangles_to_edge_index(selected, num_nodes):
    if selected.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long, device=selected.device)
    e1 = selected[:, [0, 1]]
    e2 = selected[:, [1, 2]]
    e3 = selected[:, [0, 2]]
    edges = torch.cat([e1, e2, e3], dim=0)
    edges_sorted, _ = torch.sort(edges, dim=1)
    edges_unique = torch.unique(edges_sorted, dim=0)
    return to_undirected(edges_unique.t().contiguous(), num_nodes=num_nodes)


# --- Training ---
def train_epoch(enc, sel, gnn, loader, opt_sel, opt_gnn,
                lambda_contr=1.0, lambda_struct=0.1, device=DEVICE):
    enc.train(); sel.train(); gnn.train()
    correct = total = 0
    loss_sum = 0.0
    for batch in loader:
        batch = batch.to(device)

        # --- Selector step ---
        opt_sel.zero_grad()
        tris, tri_f, tri_y = gather_batch_triangles(batch)
        if tris.size(0) > 0:
            g = enc(tri_f)
            p = sel(g, hard=True)
            L_contr = contrastive_triangle_loss_gc(p, tri_y)
            sel_mask = (p > 0.5).detach()
            sel_tris = tris[sel_mask]
            L_struct = structural_loss_gc(sel_tris, batch.x)
            L_sel = lambda_contr * L_contr + lambda_struct * L_struct
            L_sel.backward()
            opt_sel.step()
            ei_star = (triangles_to_edge_index(sel_tris, batch.x.size(0))
                       if sel_tris.numel() > 0 else batch.edge_index)
        else:
            ei_star = batch.edge_index

        # --- GNN step ---
        opt_gnn.zero_grad()
        logits = gnn(batch.x, ei_star, batch.batch)
        L_gnn = F.cross_entropy(logits, batch.y)
        L_gnn.backward()
        opt_gnn.step()

        with torch.no_grad():
            preds = logits.argmax(dim=1)
            correct += (preds == batch.y).sum().item()
            total += batch.y.size(0)
            loss_sum += float(L_gnn) * batch.y.size(0)
    return correct / max(total, 1), loss_sum / max(total, 1)


@torch.no_grad()
def eval_epoch(enc, sel, gnn, loader, device=DEVICE):
    enc.eval(); sel.eval(); gnn.eval()
    correct = total = 0
    for batch in loader:
        batch = batch.to(device)
        tris, tri_f, _ = gather_batch_triangles(batch)
        if tris.size(0) > 0:
            g = enc(tri_f)
            p = sel(g, hard=False)
            sel_mask = p >= 0.5
            sel_tris = tris[sel_mask]
            ei_star = (triangles_to_edge_index(sel_tris, batch.x.size(0))
                       if sel_tris.numel() > 0 else batch.edge_index)
        else:
            ei_star = batch.edge_index
        logits = gnn(batch.x, ei_star, batch.batch)
        preds = logits.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / max(total, 1)


def run_trigon_graph_classif(dataset, num_node_features, num_classes,
                             seed=0, max_epochs=200, patience=40,
                             batch_size=32, hidden=64,
                             lr_selector=1e-3, lr_gnn=1e-3,
                             lambda_contr=1.0, lambda_struct=0.1,
                             verbose=True):
    set_seed(seed)

    if not hasattr(dataset[0], "cand_triangles") or dataset[0].cand_triangles is None:
        t0 = time.time()
        precompute_candidates_for_dataset(dataset, k_knn=5, verbose=verbose)
        if verbose:
            print(f"Precompute took {time.time() - t0:.1f}s")

    labels = np.array([int(d.y) for d in dataset])
    idx = np.arange(len(dataset))
    idx_tr, idx_rest, y_tr, y_rest = train_test_split(
        idx, labels, test_size=0.2, random_state=seed, stratify=labels)
    idx_val, idx_te, _, _ = train_test_split(
        idx_rest, y_rest, test_size=0.5, random_state=seed, stratify=y_rest)
    train_set = [dataset[i] for i in idx_tr]
    val_set = [dataset[i] for i in idx_val]
    test_set = [dataset[i] for i in idx_te]

    tr_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    enc = TriangleEncoder(num_node_features, hidden=hidden).to(DEVICE)
    sel = TriangleSelector(hidden).to(DEVICE)
    gnn = TrigonGraphClassifier(num_node_features, hidden, num_classes,
                                num_layers=3).to(DEVICE)

    opt_sel = torch.optim.Adam(list(enc.parameters()) + list(sel.parameters()),
                               lr=lr_selector, weight_decay=5e-5)
    opt_gnn = torch.optim.Adam(gnn.parameters(), lr=lr_gnn, weight_decay=5e-5)

    best_val = best_test = 0.0
    ctr = 0
    for epoch in range(max_epochs):
        train_acc, train_loss = train_epoch(
            enc, sel, gnn, tr_loader, opt_sel, opt_gnn,
            lambda_contr=lambda_contr, lambda_struct=lambda_struct,
        )
        val_acc = eval_epoch(enc, sel, gnn, val_loader)
        test_acc = eval_epoch(enc, sel, gnn, te_loader)

        if val_acc > best_val:
            best_val = val_acc; best_test = test_acc; ctr = 0
        else:
            ctr += 1

        if verbose and epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | train_loss={train_loss:.3f} "
                  f"train_acc={train_acc:.3f} val={val_acc:.3f} test={test_acc:.3f} "
                  f"(best_val={best_val:.3f} best_test={best_test:.3f})")

        if ctr >= patience:
            if verbose:
                print(f"[early stop] epoch {epoch}")
            break

    return best_val, best_test


# --- Load and convert PROTEINS once ---
print("Loading PROTEINS...")
raw_dataset = TUDataset(root="data/TU_PROTEINS", name="PROTEINS")
DATASET_NUM_FEATURES = raw_dataset.num_node_features
DATASET_NUM_CLASSES = raw_dataset.num_classes
# Convert to CustomData, keeping standard attributes only.
graphs = [CustomData(x=d.x, edge_index=d.edge_index, y=d.y) for d in raw_dataset]
print(f"PROTEINS: {len(graphs)} graphs, {DATASET_NUM_CLASSES} classes, "
      f"{DATASET_NUM_FEATURES} node features.")
print(f"Avg nodes: {np.mean([d.num_nodes for d in graphs]):.1f}")
print(f"Max nodes: {np.max([d.num_nodes for d in graphs])}")

# Smoke test on one seed.
best_val, best_test = run_trigon_graph_classif(
    graphs, DATASET_NUM_FEATURES, DATASET_NUM_CLASSES,
    seed=0, max_epochs=100, patience=25, verbose=True,
)
print(f"\nSmoke-test: val={best_val:.4f} test={best_test:.4f}")

Device: cuda
Loading PROTEINS...


Processing...
Done!


PROTEINS: 1113 graphs, 2 classes, 3 node features.
Avg nodes: 39.1
Max nodes: 620
Precomputed 448636 candidate triangles across 1113 graphs (avg 403.1 per graph).
Precompute took 4.8s
Epoch 000 | train_loss=0.636 train_acc=0.634 val=0.748 test=0.732 (best_val=0.748 best_test=0.732)
Epoch 010 | train_loss=0.550 train_acc=0.749 val=0.775 test=0.777 (best_val=0.775 best_test=0.777)
Epoch 020 | train_loss=0.538 train_acc=0.746 val=0.775 test=0.786 (best_val=0.784 best_test=0.786)
Epoch 030 | train_loss=0.521 train_acc=0.755 val=0.775 test=0.759 (best_val=0.793 best_test=0.777)
Epoch 040 | train_loss=0.515 train_acc=0.761 val=0.757 test=0.777 (best_val=0.793 best_test=0.777)
Epoch 050 | train_loss=0.509 train_acc=0.751 val=0.784 test=0.759 (best_val=0.802 best_test=0.795)
Epoch 060 | train_loss=0.513 train_acc=0.758 val=0.766 test=0.795 (best_val=0.802 best_test=0.795)
Epoch 070 | train_loss=0.508 train_acc=0.749 val=0.757 test=0.786 (best_val=0.802 best_test=0.795)
[early stop] epoch 71

S

In [3]:
# Cell 16.3 — multi-seed evaluation on PROTEINS
SEEDS = list(range(10))
results = []
for s in SEEDS:
    v, t = run_trigon_graph_classif(
        graphs, DATASET_NUM_FEATURES, DATASET_NUM_CLASSES,
        seed=s, max_epochs=100, patience=25, verbose=False,
    )
    print(f"seed={s}: val={v:.4f} test={t:.4f}")
    results.append(t)

results = np.array(results)
print(f"\nPROTEINS TRIGON (graph classif): "
      f"{results.mean() * 100:.2f} +/- {results.std() * 100:.2f}")

seed=0: val=0.7928 test=0.7857
seed=1: val=0.7748 test=0.7143
seed=2: val=0.7658 test=0.7946
seed=3: val=0.7838 test=0.7232
seed=4: val=0.8018 test=0.6786
seed=5: val=0.8288 test=0.7589
seed=6: val=0.7387 test=0.7232
seed=7: val=0.7928 test=0.7589
seed=8: val=0.7928 test=0.7143
seed=9: val=0.7387 test=0.7321

PROTEINS TRIGON (graph classif): 73.84 +/- 3.39


### 16.4 Sanity baseline: plain GCN without TRIGON rewiring

To tell whether TRIGON helps at all on MUTAG, we compare against a plain GCN on the original graphs with the exact same hyperparameters (readout, classifier, depth). If TRIGON is worse or indistinguishable, the rewiring is not buying us anything on this task.

In [4]:
class BaselineGCN(nn.Module):
    """Standard GCN for graph classification (no rewiring)."""
    def __init__(self, in_dim: int, hidden: int, num_classes: int,
                 num_layers: int = 3, dropout: float = 0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden, hidden))

        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x, edge_index, batch):
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        # Graph-level representation via global pooling
        pooled = global_add_pool(h, batch)
        logits = self.classifier(pooled)
        return logits

In [5]:
# Cell 16.5 — plain GCN baseline on MUTAG

def run_plain_gcn_graph_classif(dataset, seed: int = 0,
                                hidden: int = 64, batch_size: int = 32,
                                max_epochs: int = 200, patience: int = 40,
                                lr: float = 1e-3,
                                verbose: bool = False):
    set_seed(seed)
    labels = np.array([int(d.y) for d in dataset])
    idx = np.arange(len(dataset))
    idx_train, idx_rest, y_train, y_rest = train_test_split(
        idx, labels, test_size=0.2, random_state=seed, stratify=labels
    )
    idx_val, idx_test, _, _ = train_test_split(
        idx_rest, y_rest, test_size=0.5, random_state=seed, stratify=y_rest
    )
    train_set = [dataset[i] for i in idx_train]
    val_set = [dataset[i] for i in idx_val]
    test_set = [dataset[i] for i in idx_test]

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    feat_dim = dataset[0].x.size(1)
    num_classes = int(labels.max()) + 1

    gnn = TrigonGraphClassifier(feat_dim, hidden, num_classes,
                                 num_layers=3, dropout=0.3).to(DEVICE)
    opt = torch.optim.Adam(gnn.parameters(), lr=lr, weight_decay=5e-5)

    def _eval(loader):
        gnn.eval()
        correct = total = 0
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(DEVICE)
                logits = gnn(batch.x, batch.edge_index, batch.batch)
                correct += (logits.argmax(dim=1) == batch.y).sum().item()
                total += batch.y.size(0)
        return correct / max(total, 1)

    best_val = best_test = 0.0
    ctr = 0
    for epoch in range(max_epochs):
        gnn.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            logits = gnn(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(logits, batch.y)
            loss.backward()
            opt.step()
        val_acc = _eval(val_loader)
        test_acc = _eval(test_loader)
        if val_acc > best_val:
            best_val = val_acc; best_test = test_acc; ctr = 0
        else:
            ctr += 1
            if ctr >= patience:
                break
    return best_val, best_test


baseline_results = []
for s in SEEDS:
    v, t = run_plain_gcn_graph_classif(graphs, seed=s, max_epochs=100, patience=25)
    print(f"[baseline] seed={s}: val={v:.4f} test={t:.4f}")
    baseline_results.append(t)

baseline_results = np.array(baseline_results)
print(f"\nPROTEINS plain GCN baseline: "
      f"{baseline_results.mean() * 100:.2f} +/- {baseline_results.std() * 100:.2f}")
print(f"\nCOMPARISON:")
print(f"  Plain GCN:   {baseline_results.mean() * 100:.2f} +/- {baseline_results.std() * 100:.2f}")
print(f"  TRIGON-GC:   {results.mean() * 100:.2f} +/- {results.std() * 100:.2f}")


[baseline] seed=0: val=0.7838 test=0.7946
[baseline] seed=1: val=0.7568 test=0.7054
[baseline] seed=2: val=0.7748 test=0.7589
[baseline] seed=3: val=0.7838 test=0.7411
[baseline] seed=4: val=0.7928 test=0.6696
[baseline] seed=5: val=0.8288 test=0.7411
[baseline] seed=6: val=0.7387 test=0.7768
[baseline] seed=7: val=0.7928 test=0.7679
[baseline] seed=8: val=0.7838 test=0.7321
[baseline] seed=9: val=0.7387 test=0.7946

PROTEINS plain GCN baseline: 74.82 +/- 3.74

COMPARISON:
  Plain GCN:   74.82 +/- 3.74
  TRIGON-GC:   73.84 +/- 3.39


### 16.6 Interpreting the comparison

**Results on PROTEINS (10 seeds, 80/10/10 stratified split):**

| Method        | Mean (%) | Std (%) |
|---------------|----------|---------|
| Plain GCN     | 74.82    | 3.74    |
| TRIGON-GC     | 73.84    | 3.39    |

TRIGON-GC lands **0.98 points below** the plain GCN baseline, with standard deviations of ~3.4-3.7 across 10 seeds. The 95% confidence intervals of the two means overlap entirely: on these 10 seeds, a paired-seed comparison would not reach statistical significance. The honest reading is **TRIGON-GC does not help on PROTEINS**.

### Why this is not surprising

This is consistent with what we expected before running: PROTEINS is a dataset where the original graph structure already carries most of the signal. Rewiring by triangle selection has little to add, and the extra machinery (triangle encoder, selector, two auxiliary losses) introduces optimization noise without compensating benefit. Three contributing factors:

1. **Weak supervision signal for the selector.** In node classification TRIGON's `L_contr` uses ground-truth node labels, which are strong and dense. In graph classification we fall back on pseudo-labels derived from the argmax of one-hot residue features — a much noisier target that does not align with the downstream classification objective.

2. **`L_part` is absent.** Dropping the class-wise participation regularizer removes one of the three losses that shaped TRIGON's rewiring in the node-classif setting. With only `L_contr` and `L_struct` guiding the selector, its inductive bias is weaker.

3. **PROTEINS graphs are small and well-structured.** Average ~39 nodes with biologically meaningful edges. Rewiring shines on graphs with long-range dependencies or structural bottlenecks; PROTEINS has neither in a significant amount.

### What we can and cannot conclude

- **Cannot conclude**: "TRIGON does not work for graph classification." We tested one adaptation on one dataset. A different pseudo-label scheme, a different readout, or a different dataset (one with more long-range dependencies, e.g. long-range graph benchmarks) could give a different answer.
- **Can conclude**: on PROTEINS with this adaptation, triangle-based rewiring is not distinguishable from a plain GCN baseline. The added complexity is not justified by accuracy gains.

This is a legitimate negative result and should be reported as such. Do not cherry-pick a favorable seed; do not drop the baseline comparison. The 10-seed mean ± std is the correct summary.
